In [17]:
# Bloco 1 - Importar, transformar e exibir a base de alunos com cidade e estado

# Importa a biblioteca Path para trabalhar com caminhos de arquivos
from pathlib import Path

# Importa a biblioteca pandas para trabalhar com dados em formato de tabela
import pandas as pd


def obter_pasta_do_notebook():
    """
    Retorna a pasta onde o notebook está sendo executado.
    No VS Code/Jupyter, Path.cwd() normalmente aponta para a pasta do arquivo .ipynb.
    """

    # Captura a pasta atual de execução do notebook
    pasta_atual = Path.cwd()

    # Retorna a pasta encontrada
    return pasta_atual


def importar_csv_mesma_pasta(nome_arquivo, nome_base):
    """
    Importa um arquivo CSV localizado na mesma pasta do notebook.

    Parâmetros:
        nome_arquivo: nome do arquivo CSV que será importado.
        nome_base: nome amigável da base, usado nas mensagens.

    Retorno:
        DataFrame do pandas com os dados importados.
        Caso ocorra erro, retorna None.
    """

    try:
        # Obtém a pasta onde o notebook está sendo executado
        pasta_notebook = obter_pasta_do_notebook()

        # Monta o caminho completo do arquivo CSV
        caminho_arquivo = pasta_notebook / nome_arquivo

        # Lê o arquivo CSV tentando identificar automaticamente o separador
        df = pd.read_csv(
            caminho_arquivo,
            sep=None,
            engine="python",
            encoding="utf-8-sig"
        )

        # Remove espaços extras antes e depois dos nomes das colunas
        df.columns = df.columns.str.strip()

        # Imprime mensagem informando que a importação deu certo
        print(f"Base '{nome_base}' importada com sucesso!")

        # Imprime o caminho completo do arquivo importado
        print(f"Arquivo importado: {caminho_arquivo}")

        # Imprime a quantidade de linhas do DataFrame
        print(f"Quantidade de linhas: {df.shape[0]}")

        # Imprime a quantidade de colunas do DataFrame
        print(f"Quantidade de colunas: {df.shape[1]}")

        # Imprime uma linha em branco para organizar a saída
        print()

        # Retorna o DataFrame criado
        return df

    except FileNotFoundError:
        # Imprime mensagem caso o arquivo não seja encontrado
        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")

        # Orienta o usuário a verificar se o arquivo está na pasta correta
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")

        # Retorna None para indicar que a importação falhou
        return None

    except Exception as erro:
        # Imprime mensagem caso aconteça qualquer outro erro
        print(f"Erro ao importar a base '{nome_base}'.")

        # Imprime os detalhes técnicos do erro
        print(f"Detalhes do erro: {erro}")

        # Retorna None para indicar que a importação falhou
        return None


def inserir_cidade_estado(df_alunos, df_municipios):
    """
    Insere o nome da cidade e o nome do estado na base de alunos.

    A ligação é feita entre:
        df_alunos['codigo_cidade']
        df_municipios['Código Município Completo']

    Retorno:
        DataFrame de alunos com as colunas nome_cidade e estado.
    """

    try:
        # Remove espaços extras dos nomes das colunas da base de alunos
        df_alunos.columns = df_alunos.columns.str.strip()

        # Remove espaços extras dos nomes das colunas da base de municípios
        df_municipios.columns = df_municipios.columns.str.strip()

        # Verifica se a coluna codigo_cidade existe na base de alunos
        if "codigo_cidade" not in df_alunos.columns:
            print("Erro: a coluna 'codigo_cidade' não foi encontrada na base de alunos.")
            return None

        # Define as colunas obrigatórias da base de municípios nova
        colunas_municipios_necessarias = [
            "Código Município Completo",
            "Nome_Município",
            "Nome_UF"
        ]

        # Verifica quais colunas obrigatórias não foram encontradas
        colunas_faltantes = [
            coluna
            for coluna in colunas_municipios_necessarias
            if coluna not in df_municipios.columns
        ]

        # Caso existam colunas faltantes, imprime o erro e mostra as colunas disponíveis
        if len(colunas_faltantes) > 0:
            print("Erro: algumas colunas necessárias não foram encontradas na base de municípios.")
            print("Colunas faltantes:", colunas_faltantes)
            print("\nColunas encontradas na base de municípios:")
            print(list(df_municipios.columns))
            return None

        # Cria uma cópia da base de alunos para evitar alterar a base original diretamente
        df_alunos_copia = df_alunos.copy()

        # Cria uma cópia da base de municípios para evitar alterar a base original diretamente
        df_municipios_copia = df_municipios.copy()

        # Converte o código da cidade da base de alunos para número inteiro
        df_alunos_copia["codigo_cidade"] = pd.to_numeric(
            df_alunos_copia["codigo_cidade"],
            errors="coerce"
        ).astype("Int64")

        # Converte o código do município da base de municípios para número inteiro
        df_municipios_copia["Código Município Completo"] = pd.to_numeric(
            df_municipios_copia["Código Município Completo"],
            errors="coerce"
        ).astype("Int64")

        # Seleciona apenas as colunas necessárias da base de municípios
        df_municipios_resumido = df_municipios_copia[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ].copy()

        # Remove códigos duplicados da base de municípios, mantendo o primeiro registro encontrado
        df_municipios_resumido = df_municipios_resumido.drop_duplicates(
            subset=["Código Município Completo"],
            keep="first"
        )

        # Renomeia as colunas da base de municípios para nomes mais simples
        df_municipios_resumido = df_municipios_resumido.rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )

        # Junta a base de alunos com a base de municípios usando o código da cidade
        df_alunos_transformado = df_alunos_copia.merge(
            df_municipios_resumido,
            on="codigo_cidade",
            how="left"
        )

        # Cria uma lista com a ordem atual das colunas
        colunas = list(df_alunos_transformado.columns)

        # Remove a coluna nome_cidade da posição atual
        colunas.remove("nome_cidade")

        # Remove a coluna estado da posição atual
        colunas.remove("estado")

        # Descobre a posição da coluna codigo_cidade
        posicao_codigo_cidade = colunas.index("codigo_cidade")

        # Insere nome_cidade logo após codigo_cidade
        colunas.insert(posicao_codigo_cidade + 1, "nome_cidade")

        # Insere estado logo após nome_cidade
        colunas.insert(posicao_codigo_cidade + 2, "estado")

        # Reorganiza o DataFrame com a nova ordem das colunas
        df_alunos_transformado = df_alunos_transformado[colunas]

        # Imprime mensagem de sucesso
        print("Dados transformados com sucesso!")

        # Conta quantos alunos ficaram sem cidade/estado após o cruzamento
        quantidade_sem_cidade = df_alunos_transformado["nome_cidade"].isna().sum()

        # Imprime a quantidade de alunos sem cidade/estado encontrados
        print(
            "Quantidade de alunos sem cidade/estado encontrados:",
            quantidade_sem_cidade
        )

        # Cria uma lista com os códigos de cidade que não foram encontrados
        codigos_nao_encontrados = (
            df_alunos_transformado
            .loc[df_alunos_transformado["nome_cidade"].isna(), "codigo_cidade"]
            .dropna()
            .drop_duplicates()
            .sort_values()
            .tolist()
        )

        # Verifica se existem códigos de cidade não encontrados
        if len(codigos_nao_encontrados) > 0:

            # Imprime uma mensagem explicando a lista
            print("\nCódigos de cidade não encontrados na base de municípios:")

            # Imprime a lista de códigos sem repetir
            print(codigos_nao_encontrados)

        else:

            # Imprime mensagem caso todos os códigos tenham sido encontrados
            print("\nTodos os códigos de cidade foram encontrados na base de municípios.")

        # Retorna o DataFrame transformado
        return df_alunos_transformado

    except Exception as erro:
        # Imprime mensagem caso aconteça erro durante a transformação dos dados
        print("Erro ao transformar os dados.")

        # Imprime os detalhes técnicos do erro
        print(f"Detalhes do erro: {erro}")

        # Retorna None para indicar falha
        return None


# Define o nome do arquivo CSV da base de alunos
nome_arquivo_alunos = "script_01_alunos.csv"

# Define o nome do arquivo CSV da base de municípios
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

# Importa a base de alunos para um DataFrame
df_alunos = importar_csv_mesma_pasta(nome_arquivo_alunos, "alunos")

# Importa a base de municípios para um DataFrame
df_municipios = importar_csv_mesma_pasta(nome_arquivo_municipios, "municípios")

# Verifica se as duas bases foram importadas corretamente antes de transformar
if df_alunos is not None and df_municipios is not None:

    # Cria um novo DataFrame com nome da cidade e estado inseridos
    df_alunos_transformado = inserir_cidade_estado(df_alunos, df_municipios)

    # Verifica se a transformação foi concluída corretamente
    if df_alunos_transformado is not None:

        # Imprime uma mensagem antes de mostrar a tabela final
        print("\nDados finais transformados:")

        # Exibe o DataFrame transformado na tela
        display(df_alunos_transformado)

        # Exporta o DataFrame transformado para CSV
        pasta_notebook = obter_pasta_do_notebook()
        nome_arquivo_saida = "script_01_alunos_consolidado.csv"
        caminho_arquivo_saida = pasta_notebook / nome_arquivo_saida

        df_alunos_transformado.to_csv(
            caminho_arquivo_saida,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"\nArquivo exportado com sucesso!")
        print(f"Caminho: {caminho_arquivo_saida}")


Base 'alunos' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_01_alunos.csv
Quantidade de linhas: 3761371
Quantidade de colunas: 21

Base 'municípios' importada com sucesso!
Arquivo importado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Dados transformados com sucesso!
Quantidade de alunos sem cidade/estado encontrados: 0

Todos os códigos de cidade foram encontrados na base de municípios.

Dados finais transformados:


,codigo_cidade,nome_cidade,estado,id_aluno,loc_latitude,loc_longitude,da_porteira,da_mataburro,da_colchete,da_atoleiro,...,cor,grau_responsavel,def_caminhar,def_ouvir,def_enxergar,def_mental,turno,nivel,dt_criacao,dt_alteracao
0,2311405,Quixeramobim,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,S,4.0,5.0,NaN,NaN
1,2312403,São Gonçalo do Amarante,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,3,NaN,N,N,N,N,2.0,2.0,NaN,NaN
2,2903235,Barro Alto,Bahia,4686956,NaN,NaN,N,N,N,N,...,0,-1.0,N,N,N,N,2.0,3.0,NaN,NaN
3,2313500,Trairi,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,N,2.0,2.0,NaN,NaN
4,2403202,Doutor Severiano,Rio Grande do Norte,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,N,3.0,2.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3761366,2513802,Santa Teresinha,Paraíba,5050475,NaN,NaN,N,S,N,S,...,3,-1.0,N,N,N,N,3.0,2.0,NaN,NaN
3761367,2923605,Paramirim,Bahia,259644,-13.5299,-42.2365,N,N,N,N,...,1,0.0,N,N,N,N,3.0,2.0,NaN,NaN
3761368,2923605,Paramirim,Bahia,5120639,NaN,NaN,N,N,N,N,...,2,0.0,N,N,N,N,3.0,1.0,NaN,NaN
3761369,2923605,Paramirim,Bahia,5120640,-13.5554,-42.2498,N,N,N,N,...,2,0.0,N,N,N,N,3.0,1.0,NaN,NaN



Arquivo exportado com sucesso!
Caminho: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_01_alunos_consolidado.csv


In [18]:
# Bloco 2 - Preparar a base para consolidação

# Cria uma cópia da base transformada para não alterar a original
df_consolidacao = df_alunos_transformado.copy()

# Preenche estados vazios com "Não identificado"
df_consolidacao["estado"] = df_consolidacao["estado"].fillna("Não identificado")

# Converte latitude para número
df_consolidacao["loc_latitude"] = pd.to_numeric(
    df_consolidacao["loc_latitude"],
    errors="coerce"
)

# Converte longitude para número
df_consolidacao["loc_longitude"] = pd.to_numeric(
    df_consolidacao["loc_longitude"],
    errors="coerce"
)

# Padroniza a coluna de porteira para texto maiúsculo
df_consolidacao["da_porteira"] = df_consolidacao["da_porteira"].astype(str).str.strip().str.upper()

# Padroniza a coluna de mata-burro para texto maiúsculo
df_consolidacao["da_mataburro"] = df_consolidacao["da_mataburro"].astype(str).str.strip().str.upper()

# Imprime mensagem de sucesso
print("Base preparada para consolidação.")

# Exibe uma prévia da base preparada
display(df_consolidacao.head())

Base preparada para consolidação.


,codigo_cidade,nome_cidade,estado,id_aluno,loc_latitude,loc_longitude,da_porteira,da_mataburro,da_colchete,da_atoleiro,...,cor,grau_responsavel,def_caminhar,def_ouvir,def_enxergar,def_mental,turno,nivel,dt_criacao,dt_alteracao
0,2311405,Quixeramobim,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,S,4.0,5.0,NaN,NaN
1,2312403,São Gonçalo do Amarante,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,3,NaN,N,N,N,N,2.0,2.0,NaN,NaN
2,2903235,Barro Alto,Bahia,4686956,NaN,NaN,N,N,N,N,...,0,-1.0,N,N,N,N,2.0,3.0,NaN,NaN
3,2313500,Trairi,Ceará,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,N,2.0,2.0,NaN,NaN
4,2403202,Doutor Severiano,Rio Grande do Norte,680,NaN,NaN,NaN,NaN,NaN,NaN,...,1,NaN,N,N,N,N,3.0,2.0,NaN,NaN


In [19]:
# Bloco 3 - Consolidar alunos com e sem georreferenciamento por estado

# Cria uma coluna booleana indicando se o aluno tem latitude e longitude preenchidas
df_consolidacao["tem_georreferenciamento"] = (
    df_consolidacao["loc_latitude"].notna() &
    df_consolidacao["loc_longitude"].notna()
)

# Agrupa por estado e calcula total, com georreferenciamento e sem georreferenciamento
tabela_georreferenciamento = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_alunos=("id_aluno", "count"),
        com_georreferenciamento=("tem_georreferenciamento", "sum"),
        sem_georreferenciamento=("tem_georreferenciamento", lambda x: (~x).sum())
    )
    .reset_index()
)

# Ordena a tabela pelo nome do estado
tabela_georreferenciamento = tabela_georreferenciamento.sort_values("estado")

# Imprime mensagem explicativa
print("Tabela consolidada de georreferenciamento por estado:")

# Exibe a tabela final
display(tabela_georreferenciamento)

Tabela consolidada de georreferenciamento por estado:


,estado,total_alunos,com_georreferenciamento,sem_georreferenciamento
0,Acre,7330,366,6964
1,Alagoas,155195,3726,151469
2,Amapá,4384,0,4384
3,Amazonas,102411,1710,100701
4,Bahia,511303,16548,494755
5,Ceará,267226,6875,260351
6,Distrito Federal,4683,9,4674
7,Espírito Santo,34976,8617,26359
8,Goiás,82083,14057,68026
9,Maranhão,144662,6095,138567


In [20]:
# Bloco 5 - Consolidar alunos com e sem mata-burro por estado

# Cria uma coluna booleana indicando se o aluno tem mata-burro
df_consolidacao["tem_mataburro"] = df_consolidacao["da_mataburro"] == "S"

# Agrupa por estado e calcula total, com mata-burro e sem mata-burro
tabela_mataburro = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_alunos=("id_aluno", "count"),
        com_mataburro=("tem_mataburro", "sum"),
        sem_mataburro=("tem_mataburro", lambda x: (~x).sum())
    )
    .reset_index()
)

# Ordena a tabela pelo nome do estado
tabela_mataburro = tabela_mataburro.sort_values("estado")

# Imprime mensagem explicativa
print("Tabela consolidada de mata-burro por estado:")

# Exibe a tabela final
display(tabela_mataburro)

Tabela consolidada de mata-burro por estado:


,estado,total_alunos,com_mataburro,sem_mataburro
0,Acre,7330,0,7330
1,Alagoas,155195,44,155151
2,Amapá,4384,0,4384
3,Amazonas,102411,1,102410
4,Bahia,511303,2791,508512
5,Ceará,267226,4167,263059
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,49,34927
8,Goiás,82083,4015,78068
9,Maranhão,144662,1593,143069


In [21]:
# Bloco 6 - Funções para consolidar colunas por estado

def consolidar_coluna_sim_nao_por_estado(df, coluna, nome_tabela):
    """
    Consolida uma coluna do tipo S/N por estado.

    Considera:
        S = com a característica
        Qualquer outro valor, inclusive N ou vazio = sem a característica
    """

    # Cria uma cópia da base para não alterar o DataFrame original
    df_temp = df.copy()

    # Padroniza a coluna como texto, remove espaços e coloca em maiúsculo
    df_temp[coluna] = df_temp[coluna].astype(str).str.strip().str.upper()

    # Cria uma coluna booleana indicando se o valor é S
    df_temp[f"tem_{coluna}"] = df_temp[coluna] == "S"

    # Agrupa por estado e calcula total, com e sem a característica
    tabela = (
        df_temp
        .groupby("estado")
        .agg(
            total_alunos=("id_aluno", "count"),
            com=("tem_" + coluna, "sum"),
            sem=("tem_" + coluna, lambda x: (~x).sum())
        )
        .reset_index()
    )

    # Renomeia as colunas para ficarem mais claras
    tabela = tabela.rename(
        columns={
            "com": f"com_{coluna}",
            "sem": f"sem_{coluna}"
        }
    )

    # Ordena a tabela pelo nome do estado
    tabela = tabela.sort_values("estado")

    # Imprime o nome da tabela
    print(nome_tabela)

    # Exibe a tabela consolidada
    display(tabela)

    # Retorna a tabela criada
    return tabela


def consolidar_coluna_preenchida_por_estado(df, coluna, nome_tabela):
    """
    Consolida uma coluna por estado verificando se ela está preenchida ou vazia.

    Considera:
        preenchida = valor informado
        vazia = valor ausente
    """

    # Cria uma cópia da base para não alterar o DataFrame original
    df_temp = df.copy()

    # Cria uma coluna booleana indicando se a coluna está preenchida
    df_temp[f"{coluna}_preenchida"] = df_temp[coluna].notna()

    # Agrupa por estado e calcula total, preenchidos e vazios
    tabela = (
        df_temp
        .groupby("estado")
        .agg(
            total_alunos=("id_aluno", "count"),
            preenchido=(f"{coluna}_preenchida", "sum"),
            vazio=(f"{coluna}_preenchida", lambda x: (~x).sum())
        )
        .reset_index()
    )

    # Renomeia as colunas para ficarem mais claras
    tabela = tabela.rename(
        columns={
            "preenchido": f"{coluna}_preenchido",
            "vazio": f"{coluna}_vazio"
        }
    )

    # Ordena a tabela pelo nome do estado
    tabela = tabela.sort_values("estado")

    # Imprime o nome da tabela
    print(nome_tabela)

    # Exibe a tabela consolidada
    display(tabela)

    # Retorna a tabela criada
    return tabela

In [22]:
# Bloco 7 - Consolidar dificuldades de acesso por estado

# Consolida alunos com e sem porteira por estado
tabela_porteira = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_porteira",
    "Tabela consolidada de porteira por estado:"
)

# Consolida alunos com e sem mata-burro por estado
tabela_mataburro = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_mataburro",
    "Tabela consolidada de mata-burro por estado:"
)

# Consolida alunos com e sem colchete por estado
tabela_colchete = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_colchete",
    "Tabela consolidada de colchete por estado:"
)

# Consolida alunos com e sem atoleiro por estado
tabela_atoleiro = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_atoleiro",
    "Tabela consolidada de atoleiro por estado:"
)

# Consolida alunos com e sem ponte rústica por estado
tabela_ponterustica = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "da_ponterustica",
    "Tabela consolidada de ponte rústica por estado:"
)

Tabela consolidada de porteira por estado:


,estado,total_alunos,com_da_porteira,sem_da_porteira
0,Acre,7330,0,7330
1,Alagoas,155195,276,154919
2,Amapá,4384,0,4384
3,Amazonas,102411,11,102400
4,Bahia,511303,1800,509503
5,Ceará,267226,1524,265702
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,60,34916
8,Goiás,82083,4144,77939
9,Maranhão,144662,851,143811


Tabela consolidada de mata-burro por estado:


,estado,total_alunos,com_da_mataburro,sem_da_mataburro
0,Acre,7330,0,7330
1,Alagoas,155195,44,155151
2,Amapá,4384,0,4384
3,Amazonas,102411,1,102410
4,Bahia,511303,2791,508512
5,Ceará,267226,4167,263059
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,49,34927
8,Goiás,82083,4015,78068
9,Maranhão,144662,1593,143069


Tabela consolidada de colchete por estado:


,estado,total_alunos,com_da_colchete,sem_da_colchete
0,Acre,7330,0,7330
1,Alagoas,155195,234,154961
2,Amapá,4384,0,4384
3,Amazonas,102411,0,102411
4,Bahia,511303,788,510515
5,Ceará,267226,357,266869
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,35,34941
8,Goiás,82083,2437,79646
9,Maranhão,144662,849,143813


Tabela consolidada de atoleiro por estado:


,estado,total_alunos,com_da_atoleiro,sem_da_atoleiro
0,Acre,7330,220,7110
1,Alagoas,155195,2346,152849
2,Amapá,4384,0,4384
3,Amazonas,102411,257,102154
4,Bahia,511303,6351,504952
5,Ceará,267226,7602,259624
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,368,34608
8,Goiás,82083,3525,78558
9,Maranhão,144662,9020,135642


Tabela consolidada de ponte rústica por estado:


,estado,total_alunos,com_da_ponterustica,sem_da_ponterustica
0,Acre,7330,3,7327
1,Alagoas,155195,268,154927
2,Amapá,4384,0,4384
3,Amazonas,102411,215,102196
4,Bahia,511303,1744,509559
5,Ceará,267226,373,266853
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,142,34834
8,Goiás,82083,2696,79387
9,Maranhão,144662,2000,142662


In [23]:
# Bloco 8 - Consolidar deficiências por estado

# Consolida alunos com e sem deficiência para caminhar por estado
tabela_def_caminhar = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "def_caminhar",
    "Tabela consolidada de deficiência para caminhar por estado:"
)

# Consolida alunos com e sem deficiência para ouvir por estado
tabela_def_ouvir = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "def_ouvir",
    "Tabela consolidada de deficiência para ouvir por estado:"
)

# Consolida alunos com e sem deficiência para enxergar por estado
tabela_def_enxergar = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "def_enxergar",
    "Tabela consolidada de deficiência para enxergar por estado:"
)

# Consolida alunos com e sem deficiência mental por estado
tabela_def_mental = consolidar_coluna_sim_nao_por_estado(
    df_consolidacao,
    "def_mental",
    "Tabela consolidada de deficiência mental por estado:"
)

Tabela consolidada de deficiência para caminhar por estado:


,estado,total_alunos,com_def_caminhar,sem_def_caminhar
0,Acre,7330,14,7316
1,Alagoas,155195,148,155047
2,Amapá,4384,0,4384
3,Amazonas,102411,201,102210
4,Bahia,511303,1338,509965
5,Ceará,267226,450,266776
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,40,34936
8,Goiás,82083,75,82008
9,Maranhão,144662,373,144289


Tabela consolidada de deficiência para ouvir por estado:


,estado,total_alunos,com_def_ouvir,sem_def_ouvir
0,Acre,7330,3,7327
1,Alagoas,155195,80,155115
2,Amapá,4384,0,4384
3,Amazonas,102411,90,102321
4,Bahia,511303,533,510770
5,Ceará,267226,133,267093
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,4,34972
8,Goiás,82083,19,82064
9,Maranhão,144662,156,144506


Tabela consolidada de deficiência para enxergar por estado:


,estado,total_alunos,com_def_enxergar,sem_def_enxergar
0,Acre,7330,8,7322
1,Alagoas,155195,175,155020
2,Amapá,4384,0,4384
3,Amazonas,102411,142,102269
4,Bahia,511303,1479,509824
5,Ceará,267226,177,267049
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,16,34960
8,Goiás,82083,40,82043
9,Maranhão,144662,239,144423


Tabela consolidada de deficiência mental por estado:


,estado,total_alunos,com_def_mental,sem_def_mental
0,Acre,7330,112,7218
1,Alagoas,155195,1546,153649
2,Amapá,4384,1,4383
3,Amazonas,102411,1012,101399
4,Bahia,511303,12389,498914
5,Ceará,267226,5094,262132
6,Distrito Federal,4683,0,4683
7,Espírito Santo,34976,351,34625
8,Goiás,82083,749,81334
9,Maranhão,144662,2046,142616


In [24]:
# Bloco 9 - Consolidar registros com e sem data de alteração por estado

# Consolida alunos com e sem data de alteração preenchida por estado
tabela_dt_alteracao = consolidar_coluna_preenchida_por_estado(
    df_consolidacao,
    "dt_alteracao",
    "Tabela consolidada de data de alteração por estado:"
)

Tabela consolidada de data de alteração por estado:


,estado,total_alunos,dt_alteracao_preenchido,dt_alteracao_vazio
0,Acre,7330,1449,5881
1,Alagoas,155195,10607,144588
2,Amapá,4384,829,3555
3,Amazonas,102411,8504,93907
4,Bahia,511303,53303,458000
5,Ceará,267226,22308,244918
6,Distrito Federal,4683,389,4294
7,Espírito Santo,34976,647,34329
8,Goiás,82083,3597,78486
9,Maranhão,144662,21463,123199


In [25]:
# Bloco 10 - Consolidar todas as tabelas e exportar em CSV e Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):
    """
    Junta todas as tabelas consolidadas por estado em uma única tabela.

    Mantém a coluna total_alunos apenas uma vez.
    """

    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):
        tabela_temp = tabela.copy()

        if indice > 0 and "total_alunos" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_alunos"])

        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    tabela_final = tabela_final.sort_values("estado").reset_index(drop=True)

    return tabela_final


tabelas_para_consolidar = [
    tabela_georreferenciamento,
    tabela_porteira,
    tabela_mataburro,
    tabela_colchete,
    tabela_atoleiro,
    tabela_ponterustica,
    tabela_def_caminhar,
    tabela_def_ouvir,
    tabela_def_enxergar,
    tabela_def_mental,
    tabela_dt_alteracao
]

tabela_consolidada_final = consolidar_tabelas_por_estado(tabelas_para_consolidar)

pasta_notebook = obter_pasta_do_notebook()

# nome_arquivo_csv = "Consolidação 01 - script_01_alunos.csv"
nome_arquivo_excel = "Consolidação 01 - script_01_alunos.xlsx"

# caminho_arquivo_csv = pasta_notebook / nome_arquivo_csv
caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

# tabela_consolidada_final.to_csv(
#     caminho_arquivo_csv,
#     index=False,
#     encoding="utf-8-sig"
# )

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")
# print(f"Arquivo CSV exportado: {caminho_arquivo_csv}")
print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)

Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: d:\Matheus\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 01 - script_01_alunos.xlsx


,estado,total_alunos,com_georreferenciamento,sem_georreferenciamento,com_da_porteira,sem_da_porteira,com_da_mataburro,sem_da_mataburro,com_da_colchete,sem_da_colchete,...,com_def_caminhar,sem_def_caminhar,com_def_ouvir,sem_def_ouvir,com_def_enxergar,sem_def_enxergar,com_def_mental,sem_def_mental,dt_alteracao_preenchido,dt_alteracao_vazio
0,Acre,7330,366,6964,0,7330,0,7330,0,7330,...,14,7316,3,7327,8,7322,112,7218,1449,5881
1,Alagoas,155195,3726,151469,276,154919,44,155151,234,154961,...,148,155047,80,155115,175,155020,1546,153649,10607,144588
2,Amapá,4384,0,4384,0,4384,0,4384,0,4384,...,0,4384,0,4384,0,4384,1,4383,829,3555
3,Amazonas,102411,1710,100701,11,102400,1,102410,0,102411,...,201,102210,90,102321,142,102269,1012,101399,8504,93907
4,Bahia,511303,16548,494755,1800,509503,2791,508512,788,510515,...,1338,509965,533,510770,1479,509824,12389,498914,53303,458000
5,Ceará,267226,6875,260351,1524,265702,4167,263059,357,266869,...,450,266776,133,267093,177,267049,5094,262132,22308,244918
6,Distrito Federal,4683,9,4674,0,4683,0,4683,0,4683,...,0,4683,0,4683,0,4683,0,4683,389,4294
7,Espírito Santo,34976,8617,26359,60,34916,49,34927,35,34941,...,40,34936,4,34972,16,34960,351,34625,647,34329
8,Goiás,82083,14057,68026,4144,77939,4015,78068,2437,79646,...,75,82008,19,82064,40,82043,749,81334,3597,78486
9,Maranhão,144662,6095,138567,851,143811,1593,143069,849,143813,...,373,144289,156,144506,239,144423,2046,142616,21463,123199
